---
title: "Part 4a: NumPy Fundamentals"
---


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/06-numpy.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/06-numpy.ipynb)

**DS-MLOps Python Foundations**

**Python 3.12+ | Author: Anthony Faustine**

You have 2,400 exam scores. You need to subtract the mean and divide by the standard deviation -- across three columns at once. A Python loop would work. It would also be thirty lines, ten seconds, and nothing like the code your teammates expect to read.

NumPy solves this with one expression: `(X - X.mean(axis=0)) / X.std(axis=0)`. No loop. No temporary variables. Just the operation, stated directly.

Part 3b built the full Python vocabulary: classes, exceptions, and file I/O. This notebook applies that vocabulary to arrays -- the data structure that underlies every ML library you will use. Part 4b (`07-numpy-advanced.ipynb`) continues with broadcasting, vectorisation, and linear algebra.

> Callout markers: [book cover page](../../index.qmd#callout-guide).

::: {.callout-note collapse="true" icon=false}
## Before You Begin

This notebook follows Parts 1-3. You should be comfortable with functions, list comprehensions, and `@dataclass`. NumPy is imported as `import numpy as np` throughout.
:::

## Meet NumPy

You have written Python for three chapters. You can loop, slice, and unpack. Now imagine you need to normalise 2,400 exam scores: subtract the mean, divide by the standard deviation, do it across three columns. A Python `for` loop would work. It would also be slow, verbose, and nothing like the code your colleagues expect to read.

NumPy ([numpy.org](https://numpy.org)) was created in 2005 by Travis Oliphant to give Python scientists the numeric performance of Fortran and C without leaving the language. The idea was simple: store numbers in a contiguous block of memory with a fixed type, and let a thin Python wrapper call heavily optimised C and Fortran routines on that block. Two decades later, nearly every numerical computing library in Python (pandas, scikit-learn, PyTorch, TensorFlow) uses NumPy arrays as its currency.

### How it compares

| Approach | Speed on large arrays | Readable math | When to use |
| --- | --- | --- | --- |
| Python `list` + loop | Slow (Python objects, GIL) | Verbose | Small collections, mixed types |
| **NumPy `ndarray`** | **Fast (C/Fortran, contiguous)** | **Concise (`a * 2`)** | **Numeric data of any size** |
| PyTorch `Tensor` | Fast (optionally GPU) | Similar to NumPy | Deep learning, autodiff |
| JAX `Array` | Very fast (XLA, JIT, GPU/TPU) | NumPy-compatible | Research, differentiable programs |
| CuPy `ndarray` | GPU only | NumPy-compatible | Large-scale GPU computing |

For everything up to classical ML on a laptop, NumPy is the right level of abstraction. PyTorch and JAX add complexity (device management, gradient tracking) that you don't need yet.

### Already in your environment

NumPy is included in `pyproject.toml`. If you ever start a standalone project:

```bash
uv add numpy          # or: pip install numpy
```

Official docs and API reference: [numpy.org/doc](https://numpy.org/doc/stable/)

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

| # | Skill | Covered in |
|---|---|---|
| 1 | Explain why NumPy arrays outperform Python lists for numeric data | Sec. 1 |
| 2 | Create arrays with `array`, `arange`, `zeros`, `ones`, and a `Generator` | Sec. 2 |
| 3 | Inspect and reshape arrays using `shape`, `dtype`, `reshape`, and stacking | Sec. 3 |
| 4 | Select data with integer indexing, slicing, and fancy indexing | Sec. 4 |
| 5 | Filter arrays with boolean masks and `np.where` | Sec. 5 |
| 6 | Compute per-feature and per-sample statistics with `axis` | Sec. 6 |
:::

## 1. Why NumPy? The `ndarray`

A Python `list` can hold anything (mixed types, nested objects), which makes it flexible but slow for numeric work: every element is a separate Python object, and arithmetic on a list means a Python-level loop.

A NumPy **`ndarray`** ("n-dimensional array") is different: it stores **one fixed dtype** in a single contiguous block of memory. That uniformity lets NumPy hand the math off to compiled C/Fortran loops instead of the Python interpreter, often 10-100x faster, and with far less memory per element.

In [ ]:
import numpy as np

study_hours = [12, 5, 18, 9, 22]

# A Python list has no element-wise arithmetic: this is string repetition, not math!
print("list  * 2 :", study_hours * 2)

hours = np.array(study_hours)
print("array * 2 :", hours * 2)  # element-wise multiplication

![Memory layout comparison: Python list stores pointers to scattered heap objects, while NumPy ndarray stores uniform float64 values in a single contiguous block.](figs/memory-layout.svg){fig-alt="Two panels. Left red panel: Python list with pointer boxes and scattered PyObject heap cells. Right blue panel: NumPy ndarray with a contiguous block of float64 values labeled one cache line."}

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: ndarray = dtype + shape + contiguous memory</span><br><br>
A NumPy array is homogeneous (one <code>dtype</code> for every element) and has a fixed <code>shape</code>. Because elements sit next to each other in memory, NumPy can vectorise operations: apply one compiled loop to the whole array instead of looping in Python.<br><br>Lists are general-purpose containers; arrays are <b>numeric data structures</b>. Use a list for a heterogeneous bag of objects, an array for a column of numbers.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - ndarray vs list arithmetic</span><br><br>
<b>Goal:</b> Feel the difference between Python lists and NumPy arrays.<br><br><b>Steps:</b><br>1. Create <code>scores_list = [78, 85, 92, 91, 55]</code> and <code>scores_arr = np.array(scores_list)</code>.<br>2. Compute the z-score: <code>(scores_arr - scores_arr.mean()) / scores_arr.std()</code>.<br>3. Confirm the result has mean ~0 and std ~1 using <code>.mean()</code> and <code>.std()</code>.<br>4. Try <code>scores_list - scores_list[0]</code> and note the error.<br><br><b>Expected:</b> z-scores around <code>[-0.3, 0.3, 1.1, 0.9, -2.0]</code>.
</div>

## 2. Creating Arrays

The most direct way to create an array is `np.array()` from a Python list (or list of lists, for 2D data). For larger or synthetic data, NumPy provides dedicated creation functions so you never have to type out values by hand.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Use default_rng, not the legacy random API</span><br><br>
<code>rng = np.random.default_rng(42)</code> creates an independent, reproducible stream. The legacy <code>np.random.seed(42)</code> sets global state shared by every library in your process -- a hidden coupling that makes results hard to reproduce. Prefer <code>default_rng</code> for all new code.
</div>

In [ ]:
# 1D array: one feature for five students
study_hours = np.array([12, 5, 18, 9, 22])

# 2D array: rows = students, columns = features
# columns: [study_hours, attendance_pct, prior_gpa]
features = np.array(
    [
        [12, 85, 3.1],
        [5, 60, 2.4],
        [18, 95, 3.8],
        [9, 70, 2.9],
        [22, 98, 3.9],
    ]
)
print(features)
print(f"shape: {features.shape}")  # (5 students, 3 features)

For larger arrays, typing out every value is impractical. These functions build arrays from a rule instead of a literal list:

| Function | Produces |
|---|---|
| `np.arange(start, stop, step)` | Evenly spaced integers/floats, like `range()` |
| `np.linspace(start, stop, n)` | `n` evenly spaced floats, **inclusive** of both ends |
| `np.zeros(shape)` / `np.ones(shape)` | Array filled with `0.0` / `1.0` |
| `np.full(shape, value)` | Array filled with a constant |
| `np.eye(n)` | `n x n` identity matrix |

In [ ]:
print("arange(0, 10)       :", np.arange(0, 10))
print("arange(0, 10, 2)    :", np.arange(0, 10, 2))
print("linspace(0, 1, 5)   :", np.linspace(0, 1, 5))
print("zeros((2, 3))       :\n", np.zeros((2, 3)))
print("ones(3)             :", np.ones(3))
print("full((2, 2), 7)     :\n", np.full((2, 2), 7))

### Random Data with a `Generator`

Synthetic data and simulations need reproducible randomness. Modern NumPy (1.17+) recommends `np.random.default_rng(seed)` over the legacy `np.random.seed(...)`. A `Generator` object is self-contained, so two generators never interfere with each other's state (unlike the old global `np.random.seed`, which silently affects every call anywhere in the program).

In [ ]:
rng = np.random.default_rng(seed=42)  # one independent, reproducible stream

print("uniform [0, 1)      :", rng.random(3))
print("normal(mean=0,std=1):", rng.normal(0, 1, size=3))
print("integers [60, 100)  :", rng.integers(60, 100, size=5))

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Prefer <code>default_rng</code> over <code>np.random.seed</code></span><br><br>
<code>np.random.seed(42)</code> mutates a single <b>global</b> random state shared by your whole program. Any other code (or library) calling <code>np.random.*</code> shifts that shared state and breaks your reproducibility. <code>rng = np.random.default_rng(42)</code> gives you an isolated generator: pass it around explicitly, and your results stay reproducible no matter what else runs.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Build a Synthetic Student Dataset</span><br><br>

<b>Goal:</b> Using <code>rng = np.random.default_rng(7)</code>, generate a feature matrix <code>X</code> of shape <code>(50, 3)</code> for 50 students with columns:<br><br>
<ul>
<li><code>study_hours</code>: <code>rng.uniform(0, 25, size=50)</code></li>
<li><code>attendance_pct</code>: <code>rng.uniform(50, 100, size=50)</code></li>
<li><code>prior_gpa</code>: <code>rng.uniform(2.0, 4.0, size=50)</code></li>
</ul>
Combine the three 1D arrays into one <code>(50, 3)</code> array with <code>np.column_stack</code>.
<pre style='background:#FCE8DA;padding:10px;border-radius:4px;font-size:0.9em'>X.shape  # -> (50, 3)
X[0]     # -> array([study_hours_0, attendance_pct_0, prior_gpa_0])</pre>
<b>Hint:</b> <code>np.column_stack([a, b, c])</code> stacks 1D arrays as columns of a 2D array.
</div>

In [ ]:
rng = np.random.default_rng(7)

study_hours = rng.uniform(0, 25, size=50)
attendance_pct = rng.uniform(50, 100, size=50)
prior_gpa = rng.uniform(2.0, 4.0, size=50)

X = ...  # TODO: combine the three arrays into one (50, 3) feature matrix

# print(f"X.shape : {X.shape}")
# print(f"X[0]    : {X[0]}")

## 3. Shape, Size, and dtype

Every array carries metadata you should check before trusting a computation: `shape` (size along each dimension), `ndim` (number of dimensions), `size` (total element count), and `dtype` (the single data type of every element).

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Check .shape and .dtype before any computation</span><br><br>
Shape mismatches and dtype surprises are the most common silent NumPy bugs. A <code>(5, 3)</code> array minus a <code>(3, 5)</code> array either errors or broadcasts in a way you did not intend. Print <code>arr.shape, arr.dtype</code> whenever a result looks wrong.
</div>

In [ ]:
X = np.array(
    [
        [12.0, 85.0, 3.1],
        [5.0, 60.0, 2.4],
        [18.0, 95.0, 3.8],
        [9.0, 70.0, 2.9],
    ]
)

print(f"shape : {X.shape}")  # (rows, columns)
print(f"ndim  : {X.ndim}")
print(f"size  : {X.size}")  # rows * columns
print(f"dtype : {X.dtype}")

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Mixed int/float input silently upcasts</span><br><br>
<code>np.array([1, 2, 3.5])</code> produces a <code>float64</code> array, not a mix of <code>int</code> and <code>float</code>. NumPy must pick <b>one</b> dtype for the whole array and silently widens every element to fit. This is usually harmless, but <code>np.array([1, 2, 3], dtype=np.int32) / 2</code> truncating, or an unexpected <code>int8</code> overflowing past 127, are the same root cause: always check <code>.dtype</code> when results look wrong.
</div>

### Reshaping

`reshape()` returns the **same data** viewed with a different shape. It doesn't copy or reorder values, so the total element count (`size`) must stay the same. `-1` tells NumPy "infer this dimension from the others":

In [ ]:
x = np.arange(12)
print(f"x          : {x}  shape={x.shape}")

x_grid = x.reshape(3, 4)
print(f"reshape(3,4):\n{x_grid}")

# -1 means "figure this dimension out for me"
x_col = x.reshape(-1, 1)  # turn a 1D array into a single column
print(f"reshape(-1,1) shape: {x_col.shape}")

x_flat = x_grid.flatten()  # back to 1D - always returns a COPY
print(f"flatten()   : {x_flat}")

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: <code>reshape()</code> returns a view, <code>flatten()</code> returns a copy</span><br><br>
Mutating the result of <code>.reshape()</code> mutates the <b>original</b> array too. They share the same underlying memory. <code>.flatten()</code> always copies, so mutating it's safe. This distinction (view vs. copy) comes up constantly in NumPy; Sec. 11 covers it in more depth.
</div>

In [ ]:
original = np.arange(6)
view = original.reshape(2, 3)
view[0, 0] = 99  # mutating the reshaped VIEW...

print(f"view     :\n{view}")
print(f"original : {original}")  # ...also changed the original!

### Combining Arrays: `column_stack`, `hstack`, `vstack`

Feature engineering often means assembling separate 1D feature vectors into one 2D matrix, or stacking two matrices together. Use the right function for the shape change you want:

| Function | Effect |
|---|---|
| `np.column_stack([a, b, ...])` | 1D arrays -> columns of a 2D array |
| `np.hstack([a, b])` | Join side-by-side (same number of rows) |
| `np.vstack([a, b])` | Stack on top of each other (same number of columns) |
| `np.concatenate([a, b], axis=...)` | General join along a chosen axis |

In [ ]:
gpa = np.array([3.1, 2.4, 3.8, 2.9])
attendance = np.array([85, 60, 95, 70])

# Two 1D feature vectors -> one (4, 2) matrix
combined = np.column_stack([gpa, attendance])
print(f"column_stack:\n{combined}")

# Two (4, 2) batches of students -> one (8, 2) matrix
more_students = np.array([[3.5, 90], [2.0, 55]])
all_students = np.vstack([combined, more_students])
print(f"vstack shape: {all_students.shape}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Build a feature matrix</span><br><br>
<b>Goal:</b> Stack 1D arrays into a 2D feature matrix and inspect its metadata.<br><br><code>study_hours  = np.array([12, 5, 18, 9, 22])</code><br><code>attendance   = np.array([85, 60, 95, 70, 98])</code><br><code>gpa          = np.array([3.1, 2.4, 3.8, 2.9, 3.9])</code><br><br><b>Steps:</b><br>1. Stack them into a <code>(5, 3)</code> matrix with <code>np.column_stack</code>.<br>2. Print <code>.shape</code> and <code>.dtype</code>.<br>3. Reshape to <code>(3, 5)</code> and confirm shapes swapped.<br><br><b>Expected:</b> original shape <code>(5, 3)</code>, reshaped <code>(3, 5)</code>.
</div>

## 4. Indexing and Slicing

NumPy indexing extends Python's list slicing to multiple dimensions. For a 2D array, the convention is `array[rows, columns]`, and negative indices still count from the end.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Basic slices return views, not copies</span><br><br>
Writing <code>sub = X[:2]</code> doesn't copy data -- <code>sub</code> shares memory with <code>X</code>. Modifying <code>sub[0, 0] = 99</code> also changes <code>X[0, 0]</code>. Call <code>.copy()</code> when you need an independent array.
</div>

In [ ]:
scores = np.array([62, 78, 85, 91, 55, 73, 88, 95, 67, 80])

print(f"first three   : {scores[:3]}")
print(f"last three    : {scores[-3:]}")
print(f"between 3 & 7 : {scores[3:7]}")
print(f"every other   : {scores[::2]}")
print(f"reversed      : {scores[::-1]}")

In [ ]:
X = np.array(
    [
        [12, 85, 3.1],
        [5, 60, 2.4],
        [18, 95, 3.8],
        [9, 70, 2.9],
        [22, 98, 3.9],
    ]
)

print(f"row 0              : {X[0]}")
print(f"column 1 (all rows): {X[:, 1]}")  # every row, attendance column
print(f"rows 1-3, col 0 & 2: \n{X[1:4, [0, 2]]}")  # fancy column indexing
print(f"single cell [2, 1] : {X[2, 1]}")

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Basic slices are views; fancy/boolean indexing copies</span><br><br>
<code>X[:, 1]</code> (a slice) returns a <b>view</b>: mutating it mutates <code>X</code>. <code>X[1:4, [0, 2]]</code> (a list of indices, known as "fancy indexing") always returns a <b>copy</b>. If you need an independent array from a slice, call <code>.copy()</code> explicitly: <code>col = X[:, 1].copy()</code>.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Select Top Performers</span><br><br>

<b>Goal:</b> Given the <code>X</code> feature matrix above (columns: study_hours, attendance_pct, prior_gpa), use slicing to print:<br><br>
<ol>
<li>The <code>prior_gpa</code> column (all rows)</li>
<li>The first two rows, all columns</li>
<li>The <code>study_hours</code> and <code>prior_gpa</code> columns (skip attendance) for every row</li>
</ol>
<b>Hint:</b> For (3), use fancy column indexing: <code>X[:, [0, 2]]</code>.
</div>

In [ ]:
X = np.array(
    [
        [12, 85, 3.1],
        [5, 60, 2.4],
        [18, 95, 3.8],
        [9, 70, 2.9],
        [22, 98, 3.9],
    ]
)

gpa_column = ...  # TODO
first_two_rows = ...  # TODO
hours_and_gpa = ...  # TODO

print(f"gpa_column     : {gpa_column}")
print(f"first_two_rows :\n{first_two_rows}")
print(f"hours_and_gpa  :\n{hours_and_gpa}")

## 5. Boolean Masking & Vectorised Conditionals

Comparing an array to a value produces a **boolean array** of the same shape: a "mask." Using that mask to index the original array keeps only the `True` positions. This replaces `if`/`for` filtering loops entirely.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Use & and | for array logic, not and / or</span><br><br>
<code>scores >= 70 and attendance >= 80</code> raises <code>ValueError</code> on arrays. Write <code>(scores >= 70) & (attendance >= 80)</code>. The parentheses matter because <code>&</code> binds tighter than <code>>=</code>.
</div>

In [ ]:
scores = np.array([62, 78, 85, 91, 55, 73, 88, 95, 67, 80])

passing_mask = scores >= 70
print(f"mask     : {passing_mask}")
print(f"passing  : {scores[passing_mask]}")  # boolean indexing: keeps True positions
print(f"n passing: {passing_mask.sum()}")  # True counts as 1, False as 0

Combine conditions with `&` (and) / `|` (or), **not** Python's `and`/`or`, which only work on single booleans, not arrays. Each side needs its own parentheses because `&`/`|` bind tighter than comparison operators:

In [ ]:
attendance = np.array([85, 60, 95, 70, 98, 45, 88, 92, 55, 80])
scores = np.array([62, 78, 85, 91, 55, 73, 88, 95, 67, 80])

# Parentheses are required: & binds tighter than >= without them
at_risk = (scores < 70) & (attendance < 70)
print(f"at_risk mask : {at_risk}")
print(f"n at risk    : {at_risk.sum()}")

`np.where(condition, if_true, if_false)` builds a new array by choosing between two values element-wise: the vectorised equivalent of a ternary expression inside a loop:

In [ ]:
labels = np.where(scores >= 70, "pass", "fail")
print(labels)

# np.select handles more than two outcomes
grade = np.select(
    [scores >= 90, scores >= 80, scores >= 70, scores >= 60],
    ["A", "B", "C", "D"],
    default="F",
)
print(grade)

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 5 - Flag Students Needing Intervention</span><br><br>

<b>Goal:</b> Given <code>scores</code> and <code>attendance</code> arrays, build a boolean mask <code>needs_help</code> that flags students with <code>score &lt; 70</code> <b>or</b> <code>attendance &lt; 60</code>, then print how many students were flagged and their scores.
<pre style='background:#FCE8DA;padding:10px;border-radius:4px;font-size:0.9em'>scores     = [62, 78, 85, 91, 55, 73, 88, 95, 67, 80]
attendance = [85, 60, 95, 70, 98, 45, 88, 92, 55, 80]
# needs_help -> True at indices 0, 4, 5, 8  (score<70 OR attendance<60)</pre>
</div>

In [ ]:
scores = np.array([62, 78, 85, 91, 55, 73, 88, 95, 67, 80])
attendance = np.array([85, 60, 95, 70, 98, 45, 88, 92, 55, 80])

needs_help = ...  # TODO: boolean mask, score < 70 OR attendance < 60

print(f"needs_help        : {needs_help}")
# print(f"n flagged         : {needs_help.sum()}")
# print(f"flagged scores    : {scores[needs_help]}")

## 6. Aggregations Along an Axis

`mean()`, `sum()`, `std()`, `min()`, `max()` collapse an array to a single number by default. On a 2D matrix, the `axis` argument controls **which dimension gets collapsed** : this is the single most common source of "right function, wrong number" bugs in DS/ML code, so get the convention straight now:

- **`axis=0`** collapses **rows** -> one result **per column** (per feature)
- **`axis=1`** collapses **columns** -> one result **per row** (per sample)

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: axis=0 collapses rows, axis=1 collapses columns</span><br><br>
<code>X.mean(axis=0)</code> gives one value per column -- per-feature statistics. <code>X.mean(axis=1)</code> gives one value per row -- per-sample statistics. Omitting <code>axis</code> collapses everything to a single scalar.
</div>

In [ ]:
X = np.array(
    [
        [12.0, 85.0, 3.1],
        [5.0, 60.0, 2.4],
        [18.0, 95.0, 3.8],
        [9.0, 70.0, 2.9],
        [22.0, 98.0, 3.9],
    ]
)

print(f"overall mean        : {X.mean():.2f}")  # one number, all 15 values
print(f"per-feature mean    : {X.mean(axis=0)}")  # shape (3,): one per column
print(f"per-student mean    : {X.mean(axis=1)}")  # shape (5,): one per row
print(f"per-feature std     : {X.std(axis=0)}")
print(f"per-feature min/max : {X.min(axis=0)} / {X.max(axis=0)}")

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Say the axis name out loud</span><br><br>
"<code>axis=0</code>" is easy to misremember. Read it as: "collapse <b>axis 0</b> (the row axis): what's left is one value per column." If you want one statistic per <i>feature</i> (the usual case before normalising a feature matrix), that is always <code>axis=0</code>.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 6 - Per-feature and per-sample statistics</span><br><br>
<b>Goal:</b> Compute aggregations along both axes.<br><br>Using the feature matrix <code>X</code> from Activity 3:<br>1. Compute the mean and std of each feature (<code>axis=0</code>).<br>2. Compute each student's mean score across all features (<code>axis=1</code>).<br>3. Z-score normalise the entire matrix with one expression:<br><code>X_z = (X - X.mean(axis=0)) / X.std(axis=0)</code><br>4. Confirm <code>X_z.mean(axis=0)</code> is all zeros (within floating-point tolerance).
</div>

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-arrow-right-circle-fill"></i> What's Next</span><br><br>
Part 4a covered the NumPy fundamentals: creating arrays, shapes, indexing, masking, and aggregations. <b>Part 4b</b> (<code>07-numpy-advanced.ipynb</code>) builds on these with the four topics that make NumPy genuinely powerful: broadcasting, vectorisation, linear algebra, and common gotchas.
</div>

## 12. Capstone Exercises

Apply everything from this notebook together. Each exercise is self-contained.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 7 - Build, Normalise, and Predict</span><br><br>

<b>Goal:</b> Using the <code>students</code> dataset below:<br><br>
<ol>
<li>Build a <code>(6, 3)</code> feature matrix <code>X</code> with <code>np.column_stack</code></li>
<li>Z-score normalise <code>X</code> (Sec. 7)</li>
<li>Predict <code>exam_score</code> with the given <code>weights</code>/<code>bias</code> using <code>@</code> (Sec. 9), applied to the <b>normalised</b> features</li>
<li>Compute the RMSE against <code>actual_scores</code> using <code>np.linalg.norm</code></li>
</ol>
</div>

In [ ]:
study_hours = np.array([12, 5, 18, 9, 22, 14])
attendance_pct = np.array([85, 60, 95, 70, 98, 80])
prior_gpa = np.array([3.1, 2.4, 3.8, 2.9, 3.9, 3.3])
actual_scores = np.array([88.0, 65.0, 95.0, 78.0, 99.0, 84.0])

weights = np.array([0.8, 0.5, 6.0])
bias = 55.0

# TODO: 1) build X, 2) normalise it, 3) predict, 4) compute RMSE
X = ...
X_normalised = ...
predicted = ...
rmse = ...

if rmse is not ...:
    print(f"predicted: {predicted}")
    print(f"RMSE     : {rmse:.2f}")
else:
    print("(complete the TODO above to see output)")

## Further Reading

| Resource | Why it matters |
|---|---|
| Harris, C.R. et al. (2020). [Array programming with NumPy](https://doi.org/10.1038/s41586-020-2649-2). *Nature* 585, 357–362. | The primary citation for NumPy; the paper explains the design decisions behind broadcasting and ufuncs |
| VanderPlas, J. (2016). *Python Data Science Handbook*, Ch. 2. O'Reilly. | Free online: the most readable treatment of fancy indexing, broadcasting, and structured arrays |
| [NumPy documentation: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) | Official broadcasting rules with diagrams; bookmark for the next time the shapes don't align |
| [NumPy documentation: Indexing](https://numpy.org/doc/stable/user/basics.indexing.html) | Covers basic, advanced, and boolean indexing in one place |


## Summary

| Concept | Key rule |
|---|---|
| `ndarray` | One dtype, contiguous memory, fast because it skips the Python interpreter |
| Creation | `np.array`, `arange`, `linspace`, `zeros`/`ones`; `np.random.default_rng(seed)` for reproducible random data |
| `shape`/`dtype` | Always check before trusting a result; mixed-type input silently upcasts |
| `reshape` | Returns a **view**, same data, same `size`, different shape |
| `flatten` | Always returns a **copy** |
| Slicing | Basic slices (`X[:, 0]`) are views; fancy/boolean indexing always copies |
| Boolean masks | `&` / `\|` (not `and`/`or`) on arrays, each side parenthesised |
| `np.where` / `np.select` | Vectorised if/else and multi-branch labelling |
| `axis=0` vs `axis=1` | 0 collapses rows -> one value per **column**; 1 collapses columns -> one value per **row** |
| Broadcasting | Compare shapes right-to-left; dims match if equal or one is `1`; use `keepdims=True` to broadcast a per-row stat |
| Vectorisation | A NumPy expression beats a Python loop by 10-100x, look for one before writing `for` |
| `@` / `np.dot` | Matrix multiplication: `X @ weights` predicts every row in one call |
| `np.allclose` | Always compare floats with a tolerance, never `==` |
| `.npy` / `.npz` | Compact, dtype/shape-preserving array storage between pipeline stages |


**Next:** `08-matplotlib.ipynb`, covering how to visualise arrays and DataFrames with matplotlib.